# UJIAN TENGAH SEMESTER (UTS) - GRAF SAINS DATA
## Tugas Pendahuluan (Setup Lingkungan & Neo4j Driver)

---
**Nama**: Rahmawati Dian Pratiwi

**NIM**: 24031554183

**Kelas**: 2024B

**Mata Kuliah**: Graf Sains Data 

---

## 1.1 Persiapan dan Perkenalan Lingkungan

### Langkah-langkah Terminal:
1. **Membuat virtual environment baru** menggunakan `venv` atau `uv`:
   ```powershell
   python -m venv myvenv
   ```
2. **Deaktivasi environment lama & aktivasi environment baru**:
   ```powershell
   .\myvenv\Scripts\Activate.ps1
   ```
3. **Instalasi paket dependensi dari `requirements.txt`**:
   ```powershell
   pip install -r requirements.txt
   ```

In [1]:
# Verifikasi pemuatan library dan versinya
import neo4j
import pandas as pd
import networkx as nx
import sklearn

print(f"Neo4j Driver Version : {neo4j.__version__}")
print(f"Pandas Version       : {pd.__version__}")
print(f"NetworkX Version     : {nx.__version__}")
print(f"Scikit-Learn Version : {sklearn.__version__}")
print("Seluruh dependensi dasar berhasil dimuat dengan baik.")

Neo4j Driver Version : 4.2.1
Pandas Version       : 2.2.3
NetworkX Version     : 3.4.2
Scikit-Learn Version : 1.6.1
Seluruh dependensi dasar berhasil dimuat dengan baik.


---
## 1.2 Mempelajari Penggunaan Driver Neo4J

Mendefinisikan koneksi ke instance Neo4j lokal pada database `183utsgraf`, membuat simpul `Person`, menghubungkan relasi `[:KNOWS]`, dan membaca data kembali.

In [10]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Membaca variabel dari file .env lokal
load_dotenv()

# Konfigurasi Koneksi Neo4J (Otomatis membaca .env jika ada, atau default placeholder aman untuk publik)
URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "password_anda")
DB_NAME = os.getenv("NEO4J_DATABASE", "183utsgraf")

AUTH = (USER, PASSWORD)

# 1. Uji Konektivitas ke Database
try:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        with driver.session(database=DB_NAME) as session:
            test_res = session.run("RETURN 'Koneksi Berhasil' AS status").single()
            print(f"Berhasil terhubung ke Neo4j Database '{DB_NAME}'! ({test_res['status']})")
except Exception as e:
    print(f"Gagal terhubung ke Neo4j: {e}")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_6092\1142195604.py:19: ExperimentalWarning: The configuration may change in the future.
  driver.verify_connectivity()


Berhasil terhubung ke Neo4j Database '183utsgraf'! (Koneksi Berhasil)


In [4]:
# 2. Fungsi menambahkan simpul Person dan relasi KNOWS (Universal session.run)
def add_friend(driver, name, friend_name):
    query = """
    MERGE (a:Person {name: $name})
    MERGE (friend:Person {name: $friend_name})
    MERGE (a)-[:KNOWS]->(friend)
    """
    with driver.session(database=DB_NAME) as session:
        session.run(query, name=name, friend_name=friend_name)

# 3. Fungsi mencetak relasi pertemanan
def print_friends(driver, name):
    query = """
    MATCH (a:Person)-[:KNOWS]->(friend) 
    WHERE a.name = $name
    RETURN friend.name AS friendName 
    ORDER BY friendName
    """
    with driver.session(database=DB_NAME) as session:
        result = session.run(query, name=name)
        print(f"Daftar relasi teman dari '{name}':")
        for record in result:
            print(f" - {record['friendName']}")

# 4. Eksekusi penambahan teman Arthur
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")


Daftar relasi teman dari 'Arthur':
 - Guinevere
 - Lancelot
 - Merlin


### Pertanyaan Pendahuluan:

**1. Apa nama database yang digunakan pada tutorial ini?**  
> **Jawaban:** Database yang digunakan bernama `183utsgraf`.

**2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instance!**  
> **Jawaban:**  
> - **Sintaks Administrasi (DDL Cypher):**  
>   - `CREATE DATABASE nama_db;` : Membuat database baru di instance.
>   - `SHOW DATABASES;` : Menampilkan seluruh database beserta status aktivasinya.
>   - `DROP DATABASE nama_db;` : Menghapus database dari instance.
> - **Perpindahan Sesi Konteks:**  
>   - Di Neo4j Browser / cypher-shell : Menggunakan perintah konteks `:use nama_db;`.
>   - Di Python Driver : Menentukan parameter `database=DB_NAME` pada pemanggilan `driver.session(database=...)`.

**3. Apa jenis graf pertemanan yang telah dibuat di sini?**  
> **Jawaban:** **Directed Graph (Graf Berarah)**, karena hubungan/relationship didefinisikan secara asimetris searah dari node sumber ke target menggunakan sintaks panah berarah: `(a)-[:KNOWS]->(friend)`.

In [5]:
# Pembersihan data dummy Arthur sebelum masuk ke data riil (Cypher Total Reset)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database=DB_NAME) as session:
        session.run("MATCH (n:Person) DETACH DELETE n")
        print("Simpul Person tutorial pendahuluan berhasil dibersihkan.")

Simpul Person tutorial pendahuluan berhasil dibersihkan.


---
## 1.3 Melakukan Proses Impor Data dari RetailRocket ke Neo4J

Dataset yang digunakan adalah **RetailRocket E-Commerce Dataset** dari Kaggle (Roman Zykov).
Fitur-fitur utama berkas `events.csv`:
1. `timestamp`: Waktu interaksi dalam format Unix Epoch (milidetik).
2. `visitorid`: ID unik pengguna / pengunjung.
3. `event`: Jenis interaksi (`view`, `addtocart`, `transaction`).
4. `itemid`: ID unik produk / barang.
5. `transactionid`: ID transaksi pembelian (hanya terisi saat event berjenis `transaction`).

In [11]:
# Verifikasi lokasi berkas events.csv
possible_paths = [
    "ecommerce-dataset/events.csv",
    "events.csv",
    "../ecommerce-dataset/events.csv",
    "data/events.csv"
]

found_path = None
for p in possible_paths:
    if os.path.exists(p):
        found_path = p
        break

if found_path:
    print(f"Berkas events.csv terdeteksi.")
    df_preview = pd.read_csv(found_path, nrows=5)
    display(df_preview)
else:
    print("Berkas events.csv belum ditemukan.")

Berkas events.csv terdeteksi.


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
